# Homework 4: Spatial and Temporal Blocking for Model Evaluation

**Goal:** Compare ordinary random cross-validation with blocked cross-validation for spatially and temporally correlated environmental data.

In this workbook, you will use a gridded **sea surface temperature (SST)** dataset from oceanography. SST varies smoothly across space and seasonally through time, so nearby observations are not independent. This is a good example for why random cross-validation can be overly optimistic.

By the end of the activity, you should be able to explain why:

1. Random k-fold cross-validation can leak information when nearby points or nearby months appear in both training and test folds.
2. Temporal block cross-validation better tests whether a model generalizes to unseen years.
3. Spatial block cross-validation better tests whether a model generalizes to unseen regions.
4. Blocked evaluation often gives worse, but more realistic, estimates of predictive performance.

**Dataset:** NOAA Extended Reconstructed Sea Surface Temperature version 5, accessed through `xarray`'s tutorial datasets. NOAA describes ERSSTv5 as a global monthly SST analysis on a 2-degree grid derived from ICOADS, with records extending from 1854 to the present. See:  
https://www.psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html  
https://docs.xarray.dev/en/stable/generated/xarray.tutorial.open_dataset.html

## Setup

Run the cell below. If any package is missing, uncomment the install line and run it once.

This notebook intentionally gives you some complete code and some incomplete code. Sections marked **TODO** are places where you should write or modify code yourself.

In [ ]:
# Uncomment if needed:
# !pip install numpy pandas xarray netcdf4 scikit-learn matplotlib

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import requests
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from skgstat import Variogram
pd.set_option("display.max_columns", 50)

## 2. Load the sea surface temperature dataset

The data are stored as an `xarray.Dataset`, which is common for gridded environmental and oceanographic data. Instead of rows and columns only, the data have dimensions such as `time`, `lat`, and `lon`.

The SST variable is measured over a regular latitude-longitude grid through time.

In [ ]:
url = "https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/sst.nc"
file_path = Path("sst.nc")

if not file_path.exists():
    r = requests.get(url)
    r.raise_for_status()
    file_path.write_bytes(r.content)

ds = xr.open_dataset(file_path)
sst = ds["sst"]
sst

### Discussion

Look at the dataset metadata above.

Questions to answer:

1. What are the dimensions of the dataset?
2. What is the spacing between latitude and longitude grid cells?
3. How many time steps are included?
4. Why might neighboring grid cells have similar SST values?
5. Why might consecutive months have similar SST values?

**Your answer:** TODO

## Explore the data visually

Before modeling, inspect the target variable. SST has strong geographic structure and seasonal/temporal structure.

The first plot shows the SST field for one time step. Nearby ocean cells usually have similar temperatures, which is spatial autocorrelation.

In [ ]:
one_time = sst.isel(time=0)

fig, ax = plt.subplots(figsize=(10, 5))
one_time.plot(ax=ax)
ax.set_title(f"Sea surface temperature on {pd.to_datetime(one_time.time.values).date()}")
plt.show()

### Explore change through time

The next plot averages SST across the selected region for each time step.

In [ ]:
regional_mean = sst.mean(dim=["lat", "lon"], skipna=True)

fig, ax = plt.subplots(figsize=(10, 4))
regional_mean.plot(ax=ax)
ax.set_title("Regional mean SST over time")
ax.set_ylabel("SST")
plt.show()

### TODO: describe the correlation structure

Write a short answer in this markdown cell after running the plots.

**Prompt:** Based on the maps and time series, where do you expect random cross-validation to be misleading? Consider both nearby locations and nearby months.

**Your answer:** TODO

>

## Convert gridded data to a machine-learning table

Most `sklearn` estimators expect a table where each row is one observation. Here, one row will represent one grid cell at one time.

The target is `sst`. The predictors will be simple features based on location and time.

We create cyclic features for month and longitude because December is close to January, and longitude wraps around the globe.

In [ ]:
df = sst.to_dataframe().reset_index().dropna(subset=["sst"]).copy()

# Time features
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["time_index"] = (df["year"] - df["year"].min()) * 12 + (df["month"] - 1)

# Cyclic month features
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Cyclic longitude features
df["lon_sin"] = np.sin(2 * np.pi * df["lon"] / 360)
df["lon_cos"] = np.cos(2 * np.pi * df["lon"] / 360)

print(df.shape)
df.head()

### Quick table checks

Use this section to understand the rows that the machine-learning model will see.

In [ ]:
print("Number of rows:", len(df))
print("Number of years:", df["year"].nunique())
print("Years:", df["year"].min(), "to", df["year"].max())
print("Number of grid locations:", df[["lat", "lon"]].drop_duplicates().shape[0])

print("\nTarget summary:")
df["sst"].describe()

### TODO: additional exploration

Add at least two exploratory checks of your own. Examples:

- Plot SST versus latitude.
- Plot SST versus month.
- Compare SST distributions for different years.
- Map a different time step.
- Compute average SST by latitude band.

Write your code in the empty cell below.

In [ ]:
# TODO: Add at least two exploratory checks of your own.

In [ ]:
# TODO: Add historic average temperture for that location be careful to not use future data when creating this column
df['Month_Average_sst'] = # Just copy what you did for InClass4

## Define the prediction problem

We will predict SST from simple location and time features.

This is not meant to be the best possible ocean forecasting model. The goal is to study how evaluation changes when data are spatially and temporally correlated.

In [ ]:
feature_cols = [
    "lat",
    "lon_sin",
    "lon_cos",
    "month_sin",
    "month_cos",
    "time_index",
    "Month_Average_sst"
]

target_col = "sst"

X = df[feature_cols]
y = df[target_col]

X.head()

## Create a reusable model and evaluation function

We use a tree-based regression model from `sklearn`. It can capture nonlinear patterns such as warmer temperatures near the equator and seasonal variation.

The helper function below runs cross-validation and returns RMSE, MAE, and R² for each fold.

Interpretation of metrics:

- **RMSE:** Typical prediction error, with larger errors penalized more strongly.
- **MAE:** Typical absolute error.
- **R²:** Fraction of variance explained. Higher is better, but it can be misleading when validation folds are not independent.

In [ ]:
def make_model():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("regressor", HistGradientBoostingRegressor(
            max_iter=100,
            learning_rate=0.08,
            max_leaf_nodes=31,
            random_state=42,
        ))
    ])


def summarize_cv(model, X, y, cv, groups=None):
    """Run cross-validation and return fold-level metrics."""
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        groups=groups,
        scoring={
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
        n_jobs=-1,
        return_train_score=False,
    )

    results = pd.DataFrame({
        "fold": np.arange(1, len(scores["test_rmse"]) + 1),
        "RMSE": -scores["test_rmse"],
        "MAE": -scores["test_mae"],
        "R2": scores["test_r2"],
    })
    return results


def summarize_mean(results, label):
    """Summarize fold-level CV results into one row."""
    return pd.DataFrame({
        "Evaluation": [label],
        "Mean RMSE": [results["RMSE"].mean()],
        "SD RMSE": [results["RMSE"].std()],
        "Mean MAE": [results["MAE"].mean()],
        "Mean R2": [results["R2"].mean()],
    })

## Naive evaluation: random k-fold cross-validation

Random k-fold cross-validation mixes observations from different locations and times into each fold.

This is common in introductory machine learning, but it is often too optimistic for spatial-temporal data because the test set can contain points very close to the training set in space and time.

Run the cell below and record the results.

In [ ]:
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)

random_results = summarize_cv(
    model=make_model(),
    X=X,
    y=y,
    cv=random_cv,
)

random_results

In [ ]:
random_summary = summarize_mean(random_results, "Random k-fold CV")
random_summary

### Discussion: random CV

Questions:

1. What is the average RMSE under random k-fold CV?
2. Why might this number be too optimistic?
3. In this dataset, what kinds of information can leak from the training folds into the test fold?

**Your answer:** TODO

## Temporal block cross-validation

Now we will hold out entire years. This asks a different question:

> Can the model generalize to years it has not seen during training?

This is a better approximation of forecasting or prediction into future time periods than random k-fold CV.

### TODO: coding task

Complete the cell below.

Hints:

- Use `GroupKFold`.
- The grouping variable should be `df["year"]`.
- Pass the groups into `summarize_cv(..., groups=...)`.

In [ ]:
# TODO: Complete temporal block cross-validation.

# Step 1: define the groups.
temporal_groups = None  # replace None with the correct column

# Step 2: create the GroupKFold splitter.
temporal_cv = None  # replace None with GroupKFold(...)

# Step 3: run blocked cross-validation.
temporal_results = None  # replace None with summarize_cv(...)

# Step 4: display the results.
temporal_results

### TODO: summarize temporal CV

Use `summarize_mean()` to create a one-row summary for temporal block CV.

In [ ]:
# TODO: Summarize your temporal CV results.

temporal_summary = None

temporal_summary

### Discussion: temporal blocking

Questions:

1. Is temporal block CV better or worse than random CV?
2. What does the difference tell you?
3. Does the model appear to generalize well to unseen years?
4. Would this be a fair test if your real goal were to predict SST next year? Why or why not?

**Your answer:** TODO

## Spatial block cross-validation

Now we will hold out entire regions. This asks:

> Can the model generalize to locations it has not seen during training?

Spatial blocking is useful when your real prediction task involves mapping unsampled locations or transferring a model to a new region.

We first create coarse spatial blocks using latitude and longitude bins.

In [ ]:
# TODO Create a variogram of residuals to determine the extent of auto correlation
# HINT: Look back at InClass3

### Create spatial blocking

Once you have determined the extend of auto correlation, use that number to set the block size.

In [ ]:
# Create coarse spatial blocks.
# You may change these sizes later and see how the results change.
lat_block_size = None # replace None based on your results above
lon_block_size = None # replace Nased on your results above

df["lat_block"] = np.floor((df["lat"] + 90) / lat_block_size).astype(int)
df["lon_block"] = np.floor(df["lon"] / lon_block_size).astype(int)
df["spatial_block"] = df["lat_block"].astype(str) + "_" + df["lon_block"].astype(str)

print("Number of spatial blocks:", df["spatial_block"].nunique())
df[["lat", "lon", "lat_block", "lon_block", "spatial_block"]].head()

### Visualize the spatial blocks

Each point below is one observation in the table, but many time steps occur at each location. The colors represent block IDs numerically, not physical values.

In [ ]:
# Make a location-level table for plotting blocks.
block_map = df[["lat", "lon", "spatial_block"]].drop_duplicates().copy()
block_map["block_code"] = pd.factorize(block_map["spatial_block"])[0]

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(block_map["lon"], block_map["lat"], c=block_map["block_code"], s=20)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Spatial blocks used for blocked cross-validation")
plt.show()

### Coding task

Complete the spatial block cross-validation.

Hints:

- Use `GroupKFold` again.
- The grouping variable should be `df["spatial_block"]`.
- Remember that `X` was created before the spatial block columns were added. That is okay because the block labels are for splitting only, not prediction.

In [ ]:
# TODO: Complete spatial block cross-validation.

# Step 1: define the groups.
spatial_groups = None  # replace None with the correct column

# Step 2: create the GroupKFold splitter.
spatial_cv = None  # replace None with GroupKFold(...)

# Step 3: run blocked cross-validation.
spatial_results = None  # replace None with summarize_cv(...)

# Step 4: display the results.
spatial_results

### Coding task: summarize spatial CV

Use `summarize_mean()` to create a one-row summary for spatial block CV.

In [ ]:
# TODO: Summarize your spatial CV results.

spatial_summary = None

spatial_summary

### Discussion: spatial blocking

Questions:

1. Is spatial block CV better or worse than random CV?
2. What does this say about the model's ability to predict in new locations?
3. Why is latitude likely such a powerful predictor of SST?
4. Why might a model still struggle when entire regions are withheld?

**Your answer:** TODO

## Spatial-temporal holdout

Spatial blocking asks whether the model can generalize to new places. Temporal blocking asks whether the model can generalize to new years. A spatial-temporal holdout combines both ideas.

Here we create a stricter test: the test set is a selected region during selected later years, and the training set excludes both the held-out region and the held-out years. This asks:

> Can the model predict a new region during a new time period?

This is usually harder than random CV, temporal block CV, or spatial block CV alone.


In [ ]:
def strict_spatiotemporal_kfold(
    df,
    spatial_col="spatial_block",
    time_col="year",
    n_splits=5,
    random_state=42,
):
    """
    Create strict spatial-temporal folds.

    For each fold:
    - Test data are from held-out spatial blocks AND held-out time blocks.
    - Training data exclude held-out spatial blocks AND exclude held-out time blocks.

    This prevents the model from seeing:
    - the same spatial region in other years
    - the same year in other spatial regions
    """

    spatial_blocks = np.array(sorted(df[spatial_col].dropna().unique()))
    time_blocks = np.array(sorted(df[time_col].dropna().unique()))

    spatial_kfold = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    time_kfold = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    spatial_folds = list(spatial_kfold.split(spatial_blocks))
    time_folds = list(time_kfold.split(time_blocks))

    for fold_id, ((_, spatial_test_idx), (_, time_test_idx)) in enumerate(
        zip(spatial_folds, time_folds),
        start=1
    ):
        heldout_spatial_blocks = spatial_blocks[spatial_test_idx]
        heldout_time_blocks = time_blocks[time_test_idx]

        spatial_test_mask = df[spatial_col].isin(heldout_spatial_blocks)
        time_test_mask = df[time_col].isin(heldout_time_blocks)

        test_mask = spatial_test_mask & time_test_mask

        train_mask = (~spatial_test_mask) & (~time_test_mask)

        train_idx = df.index[train_mask].to_numpy()
        test_idx = df.index[test_mask].to_numpy()

        yield fold_id, train_idx, test_idx, heldout_spatial_blocks, heldout_time_blocks

X = df[feature_cols]
y = df[target_col]
strict_st_results = []

for fold_id, train_idx, test_idx, heldout_space, heldout_time in strict_spatiotemporal_kfold(
    df,
    spatial_col="spatial_block",
    time_col="year",
    n_splits=5,
    random_state=42,
):
    model = make_model()

    X_train = X.loc[train_idx]
    X_test = X.loc[test_idx]

    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    strict_st_results.append({
        "fold": fold_id,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "heldout_spatial_blocks": len(heldout_space),
        "heldout_time_blocks": len(heldout_time),
        "r2": r2_score(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
    })

strict_st_results = pd.DataFrame(strict_st_results)
strict_st_results

In [ ]:
strict_st_cv_results = pd.DataFrame({
    "Evaluation": "Spatial temporal block cv",
    "Mean RMSE": [strict_st_results["rmse"].mean()],
    "SD RMSE": [strict_st_results["rmse"].std()],
    "Mean MAE": [strict_st_results["mae"].mean()],
    "Mean R2": [strict_st_results["r2"].mean()]
})

strict_st_cv_results

### Discussion: spatial-temporal holdout

The spatial-temporal holdout is the hardest evaluation in this workbook because the model is asked to predict observations from a region and years that were excluded from training. If this error is much larger than the random CV error, that is evidence that random CV was benefiting from spatial and temporal dependence.

The model still receives location-based features such as latitude and longitude. That means the model may learn broad geographic gradients, such as warmer water near the tropics, even though the specific holdout region is excluded. Whether this is appropriate depends on the scientific goal. If the goal is prediction at known coordinates, location can be a valid predictor. If the goal is to learn transferable physical relationships, you should also try a version without latitude and longitude features and compare the results.


## Compare all evaluation strategies

After you complete the temporal, spatial, and spatial-temporal sections, combine the summary tables.

In [ ]:
# combine all summaries into one table.

comparison = pd.concat(
    [
        random_summary,
        temporal_summary,
        spatial_summary,
        strict_st_cv_results,
    ],
    ignore_index=True,
)

comparison


### Plot the comparison

Complete the plotting code below. You should produce a bar chart of mean RMSE for the three evaluation strategies.

In [ ]:
# Answer key: bar chart of Mean RMSE by evaluation strategy.

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comparison["Evaluation"], comparison["Mean RMSE"])
ax.set_ylabel("Mean RMSE")
ax.set_title("Naive vs blocked cross-validation")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

# Optional: view all metrics again.
comparison


## Final reflection

Write a brief paragraph answering the following:

1. Which evaluation strategy gave the most optimistic result?
2. Which evaluation strategy gave the most conservative result?
3. Which strategy best matches a forecasting problem?
4. Which strategy best matches a spatial mapping problem?
5. What did this exercise teach you about correlated environmental data?

**Your answer:** TODO

>